In [1]:
from google.colab import files

uploaded = files.upload()

Saving en2en_sum_v4-s.zip to en2en_sum_v4-s.zip


In [2]:
!unzip en2en_sum_v4-s.zip -d en2en_sum_v4-s/
dir_str = "en2en_sum_v4-s"

Archive:  en2en_sum_v4-s.zip
   creating: en2en_sum_v4-s/en2en_sum_v4-s/
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/仲尼弟子列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/仲尼弟子列传/model.summary.txt  
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/南越列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/南越列传/model.summary.txt  
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/卫将军骠骑列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/卫将军骠骑列传/model.summary.txt  
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/司马穰苴列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/司马穰苴列传/model.summary.txt  
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/吕不韦列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/吕不韦列传/model.summary.txt  
   creating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/孟子荀卿列传/
  inflating: en2en_sum_v4-s/en2en_sum_v4-s/史记/七十列传/孟子荀卿列传/model.summary.txt  
   creatin

In [3]:
!cd en2en_sum_v4-s/
!ls ./en2en_sum_v4-s/en2en_sum_v4-s/

史记  后汉书  晋书  梁书  汉书


In [4]:
!rm -rf Classical-Chinese-Summarization
!git clone https://github.com/ctaiyi15/Classical-Chinese-Summarization.git
!ls Classical-Chinese-Summarization/data

Cloning into 'Classical-Chinese-Summarization'...
remote: Enumerating objects: 9491, done.
remote: Counting objects: 100% (942/942), done.
remote: Compressing objects: 100% (500/500), done.
remote: Total 9491 (delta 59), reused 903 (delta 32), pack-reused 8549 (from 1)
Receiving objects: 100% (9491/9491), 73.54 MiB | 16.04 MiB/s, done.
Resolving deltas: 100% (2141/2141), done.
Updating files: 100% (4665/4665), done.
dataset.json		processed  segmented	 segmented_v3  segmented_v6
mt5_en2en_segmented_v6	raw	   segmented_v2  segmented_v4


In [5]:

!pip install -U openai nest_asyncio
import asyncio
import json
import re
from pathlib import Path

import nest_asyncio
from openai import AsyncOpenAI
from google.colab import userdata
from tqdm.notebook import tqdm

input_root = Path(
    "Classical-Chinese-Summarization/data/processed/translated"
)

summary_root = Path(
    f"./{dir_str}/{dir_str}/"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.37.0
    Uninstalling openai-2.37.0:
      Successfully uninstalled openai-2.37.0


In [6]:
nest_asyncio.apply()

client = AsyncOpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

MAX_CONCURRENT_REQUESTS = 20
SEM = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

In [7]:
def complete_truncated_json(text):
    text = text.strip()

    stack = []
    in_string = False
    escape = False

    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            if stack:
                stack.pop()

    if in_string:
        text += '"'

    while stack:
        opener = stack.pop()
        if opener == "{":
            text += "}"
        elif opener == "[":
            text += "]"

    return text

def safe_json_loads(content, debug_info=None):

    raw_content = content

    text = content.strip()
    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    candidates = []

    # 1. Original cleaned text
    candidates.append(text)

    # 2. From first JSON object start, even if final } is missing
    first_brace = text.find("{")
    if first_brace != -1:
        candidates.append(text[first_brace:])

    # 3. From first JSON array start, useful for array-only outputs
    first_bracket = text.find("[")
    if first_bracket != -1:
        candidates.append(text[first_bracket:])

    # 4. Try completing missing closing syntax
    candidates.extend(
        complete_truncated_json(c)
        for c in list(candidates)
    )

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    print("\n" + "=" * 80)
    print("JSON ERROR: COULD NOT REPAIR JSON")
    print("=" * 80)

    if debug_info is not None:
        print("DEBUG INFO:")
        print(
            json.dumps(
                debug_info,
                ensure_ascii=False,
                indent=2
            )
        )

    print("\nRAW MODEL OUTPUT:")
    print(raw_content)
    print("=" * 80)

    raise json.JSONDecodeError(
        "Could not parse or repair JSON",
        raw_content,
        0
    )

async def guarded_completion(messages, temperature=0, max_tokens=10000):

    async with SEM:

        response = await client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=temperature,
            response_format={"type": "json_object"},
            max_tokens=max_tokens
        )

    return response

def build_pairs():

    pairs = []

    source_files = list(input_root.rglob("*.txt"))

    for src_path in source_files:

        rel = src_path.relative_to(input_root)

        summary_path = (
            summary_root / rel
        ).with_suffix(".summary.txt")

        if summary_path.exists():
            pairs.append((src_path, summary_path))

    return pairs


def read_file(path):

    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [10]:
from pathlib import Path
import shutil

summary_root = Path(f"{dir_str}/{dir_str}")

count = 0

for model_file in summary_root.rglob("model.summary.txt"):
    target_file = model_file.with_name("target.summary.txt")
    shutil.copy2(model_file, target_file)
    count += 1

print("Copied files:", count)



Copied files: 60


In [11]:
# Coverage/Recall Evaluation
# 1. Extract 20 important points from the original text;
# 2. Check whether each point is included in the summary

COVERAGE_SAVE_PATH = "coverage_live_results.json"
COVERAGE_FAILED_SAVE_PATH = "coverage_failed_files.json"

def load_important_points_json(content, debug_info=None):

    raw_content = content

    # First try normal JSON parsing.
    try:
        return safe_json_loads(
            content,
            debug_info=debug_info
        )["important_points"]

    except json.JSONDecodeError:
        pass

    # Clean markdown fences.
    text = content.strip()
    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    # Try simple repairs for missing closing characters.
    repair_candidates = [
        text,
        text.rstrip(",") + "\n  ]\n}",
        text.rstrip(",") + "\n]",
        text.rstrip(",") + "\n}"
    ]

    for candidate in repair_candidates:
        try:
            obj = json.loads(candidate)
            if "important_points" in obj:
                return obj["important_points"]
        except json.JSONDecodeError:
            continue

    # Last-resort salvage:
    # Extract quoted strings inside the important_points array.
    array_start = text.find("[")
    if array_start != -1:
        array_text = text[array_start + 1:]

        string_matches = re.findall(
            r'"((?:\\.|[^"\\])*)"',
            array_text,
            flags=re.DOTALL
        )

        points = []

        for s in string_matches:
            try:
                points.append(
                    json.loads(f'"{s}"')
                )
            except json.JSONDecodeError:
                points.append(s)

        if points:
            print("\n" + "=" * 80)
            print("WARNING: Salvaged important_points from malformed JSON")
            print("=" * 80)

            if debug_info is not None:
                print("DEBUG INFO:")
                print(
                    json.dumps(
                        debug_info,
                        ensure_ascii=False,
                        indent=2
                    )
                )

            print("Recovered points:", len(points))
            print("=" * 80)

            return points

    print("\n" + "=" * 80)
    print("FAILED TO PARSE OR SALVAGE important_points JSON")
    print("=" * 80)

    if debug_info is not None:
        print("DEBUG INFO:")
        print(
            json.dumps(
                debug_info,
                ensure_ascii=False,
                indent=2
            )
        )

    print("\nRAW OUTPUT:")
    print(raw_content)
    print("=" * 80)

    raise json.JSONDecodeError(
        "Could not parse important_points JSON",
        raw_content,
        0
    )

async def extract_important_points(source_text, debug_info=None):

    prompt = f"""
You are an expert evaluator of historical summaries.

Task:
Read the original historical text and identify the 20 most important factual points that a good summary should include.

Rules:
- Return exactly 20 points.
- Each point must express one important factual idea.
- Do NOT include minor details unless they are central to the text.
- Do NOT add outside knowledge.
- Use concise English.
- Preserve names and historical entities when important.

Return JSON ONLY:
{{
  "important_points": [
    "point 1",
    "point 2"
  ]
}}

ORIGINAL TEXT:
{source_text}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=5000
    )

    choice = response.choices[0]
    content = choice.message.content

    print("extract_important_points finish_reason:", choice.finish_reason)

    return load_important_points_json(
        content,
        debug_info=debug_info
    )[:20]


async def check_point_in_summary(point, summary_text, debug_info=None):

    prompt = f"""
You are evaluating summary coverage.

Task:
Determine whether the summary includes the important point.

Rules:
- ONLY use the summary.
- Judge meaning, not exact wording.
- Paraphrases count as included if the meaning is the same.
- Minor tense/aspect differences are acceptable.
- If the summary includes only part of the point, label it "partially_included".
- If the summary does not include the point, label it "not_included".
- Be strict about factual content, but not about wording.

Return JSON ONLY:
{{
  "label": "included"
}}

Allowed labels:
- "included"
- "partially_included"
- "not_included"

IMPORTANT POINT:
{point}

SUMMARY:
{summary_text}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100
    )

    content = response.choices[0].message.content

    return safe_json_loads(
        content,
        debug_info=debug_info
    )


async def process_coverage_point(point, summary_text, debug_info=None):

    verdict = await check_point_in_summary(
        point,
        summary_text,
        debug_info=debug_info
    )

    return {
        "point": point,
        "label": verdict["label"]
    }


async def evaluate_coverage(source_text, summary_text, debug_info=None):

    important_points = await extract_important_points(
        source_text,
        debug_info={
            **(debug_info or {}),
            "stage": "extract_important_points"
        }
    )

    # Defensive cleanup
    important_points = important_points[:20]

    tasks = [
        process_coverage_point(
            point,
            summary_text,
            debug_info={
                **(debug_info or {}),
                "stage": "check_point_in_summary",
                "point_index": idx,
                "total_points": len(important_points),
                "point": point
            }
        )
        for idx, point in enumerate(important_points, start=1)
    ]

    results = await asyncio.gather(*tasks)

    included = sum(
        r["label"] == "included"
        for r in results
    )

    partially_included = sum(
        r["label"] == "partially_included"
        for r in results
    )

    not_included = sum(
        r["label"] == "not_included"
        for r in results
    )

    total = len(results)

    coverage_score = (
        included + 0.5 * partially_included
    ) / total if total > 0 else 0

    return {
        "coverage_score": coverage_score,
        "total_points": total,
        "included_points": included,
        "partially_included_points": partially_included,
        "not_included_points": not_included,
        "details": results
    }


async def process_file_coverage(src_path, sum_path):

    source_text = read_file(src_path)
    summary_text = read_file(sum_path)

    debug_info = {
        "source_file": str(src_path),
        "summary_file": str(sum_path)
    }

    result = await evaluate_coverage(
        source_text,
        summary_text,
        debug_info=debug_info
    )

    return {
        "source_file": str(src_path),
        "summary_file": str(sum_path),
        **result
    }


async def process_file_coverage_with_paths(src, summ):

    try:
        result = await process_file_coverage(src, summ)

        return {
            "ok": True,
            "source_file": str(src),
            "summary_file": str(summ),
            "result": result
        }

    except Exception as e:

        return {
            "ok": False,
            "source_file": str(src),
            "summary_file": str(summ),
            "error_type": type(e).__name__,
            "error": str(e),
            "repr": repr(e)
        }


async def run_first_x_coverage_live_saved(x):

    pairs = build_pairs()[:x]

    print(f"Running coverage evaluation on {len(pairs)} file pairs")

    results = []
    errors = []

    tasks = [
        process_file_coverage_with_paths(src, summ)
        for src, summ in pairs
    ]

    completed = 0
    failed = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):

        item = await coro

        if item["ok"]:

            result = item["result"]
            results.append(result)
            completed += 1

            score = result["coverage_score"]

            avg = sum(
                r["coverage_score"]
                for r in results
            ) / len(results)

            print("\n" + "=" * 60)
            print(f"SUCCESS [{completed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("Coverage score:", score)
            print("Running avg:", avg)

            display_result = {
                k: v
                for k, v in result.items()
                if k != "details"
            }

            print(
                json.dumps(
                    display_result,
                    ensure_ascii=False,
                    indent=2
                )
            )

            with open(
                COVERAGE_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    results,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

        else:

            failed += 1
            errors.append(item)

            print("\n" + "=" * 60)
            print(f"FAILED [{failed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("ERROR:", item["repr"])

            with open(
                COVERAGE_FAILED_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    errors,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

    print("\nDONE")
    print("Successful files:", len(results))
    print("Failed files:", len(errors))

    if results:
        final_avg = sum(
            r["coverage_score"]
            for r in results
        ) / len(results)

        print("Final coverage avg:", final_avg)
    else:
        print("No successful results. Final avg cannot be computed.")

    if errors:
        print("Failed file list saved to:", COVERAGE_FAILED_SAVE_PATH)

    return results


async def run_first_coverage_debug():

    pairs = build_pairs()[:1]

    print(f"Running coverage debug on {len(pairs)} file pair")

    if not pairs:
        print("No file pairs found")
        return None

    src_path, sum_path = pairs[0]

    print("Source file:", src_path)
    print("Summary file:", sum_path)

    result = await process_file_coverage(src_path, sum_path)

    display_result = {
        k: v
        for k, v in result.items()
        if k != "details"
    }

    print(
        json.dumps(
            display_result,
            ensure_ascii=False,
            indent=2
        )
    )

    return result




In [12]:
coverage_debug_result = await run_first_x_coverage_live_saved(1000)
from google.colab import files
import shutil
shutil.copy("coverage_live_results.json", f"coverage_live_results-{dir_str}-1.json")
files.download(f"coverage_live_results-{dir_str}-1.json")

Running coverage evaluation on 60 file pairs


  0%|          | 0/60 [00:00<?, ?it/s]

extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_im

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
coverage_debug_result2 = await run_first_x_coverage_live_saved(1000)
from google.colab import files
import shutil
shutil.copy("coverage_live_results.json", f"coverage_live_results-{dir_str}-2.json")
files.download(f"coverage_live_results-{dir_str}-2.json")

Running coverage evaluation on 60 file pairs


  0%|          | 0/60 [00:00<?, ?it/s]

extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_im

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
coverage_debug_result3 = await run_first_x_coverage_live_saved(1000)
from google.colab import files
import shutil
shutil.copy("coverage_live_results.json", f"coverage_live_results-{dir_str}-3.json")
files.download(f"coverage_live_results-{dir_str}-3.json")

Running coverage evaluation on 60 file pairs


  0%|          | 0/60 [00:00<?, ?it/s]

extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_important_points finish_reason: stop
extract_im

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>